# Empirical Validation of the Macro Regime Model

**Goal.** Test whether the Growth / Inflation / Liquidity pillars carry information beyond their construction. We expect modest R² — success is *signed coefficients matching economic priors with HAC t > 2 on at least one (asset, horizon, pillar)*, plus AUC > 0.7 for the recession nowcast.

## Plan

1. **Data** — fetch FRED + yfinance (cached to `validation/_cache/`).
2. **A. Construct validity** — PCA + Cronbach's α per pillar.
3. **B. Predictive validity** — OLS of forward returns on pillar scores, Newey–West HAC errors. Sign table vs. priors.
4. **C. Recession nowcast** — logit on USREC, AUC + pseudo-R².
5. **D. Regime-conditional returns** — means / Sharpes by regime, ANOVA + Kruskal–Wallis.
6. **E. Walk-forward parity** — confirm rolling z-score is causal.
7. **F. Sensitivity** — vary z-window and threshold; flag fragile findings.

Outputs are written to `validation/outputs/`.

In [ ]:
import sys, os
from pathlib import Path

# Run from the repo root so `import regime_model` works.
REPO = Path.cwd().resolve()
if REPO.name == 'validation':
    REPO = REPO.parent
    os.chdir(REPO)
sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import regime_model as rm
from validation import data as vd
from validation import tests as vt

OUT = REPO / 'validation' / 'outputs'
OUT.mkdir(exist_ok=True)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 0. Build the panel

FRED + USREC + yfinance → regime scores + labels + forward returns at horizons {1, 3, 6, 12} months.

In [ ]:
panel = vd.build_panel()
scores = panel['scores']
labels = panel['labels']
prices = panel['prices']
fwd = panel['forward_returns']
usrec = panel['usrec']
regime_out = panel['regime']

print(f"Scores: {scores.shape}, range {scores.dropna().index.min().date()} → {scores.dropna().index.max().date()}")
print(f"Prices: {prices.shape}, assets {list(prices.columns)}")
print(f"USREC:  {usrec.shape if usrec is not None else 'missing'}")
scores.tail()

## A. Construct validity — do the five inputs in each pillar measure one thing?

If equal-weighting is to be defended, PC1 of each pillar's z-scored, sign-adjusted inputs should explain a substantial share of variance and all loadings should share a sign.

In [ ]:
construct_results = {}
for name, df in [('Growth', regime_out['growth_df']),
                 ('Inflation', regime_out['inflation_df']),
                 ('Liquidity', regime_out['liquidity_df'])]:
    cv = vt.pillar_construct_validity(df, name)
    construct_results[name] = cv
    print(f"\n{name} pillar  (n={cv['n_obs']})")
    print(f"  PC1 explains {cv['pc1_pct']:.1f}% of variance")
    print(f"  PC1 loadings (all same sign = {cv['all_same_sign']}):")
    for k, v in cv['pc1_loadings'].items():
        print(f"    {k:<22} {v:+.3f}")
    print(f"  Cronbach's α = {cv['cronbach_alpha']:.3f}")

# Save for the report.
pd.DataFrame({k: {'pc1_pct': v['pc1_pct'], 'cronbach_alpha': v['cronbach_alpha'],
                  'all_same_sign': v['all_same_sign'], 'n': v['n_obs']}
              for k, v in construct_results.items()}).T.to_csv(OUT / 'A_construct_validity.csv')

In [ ]:
# Per-pillar correlation heatmaps
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (name, df) in zip(axes, [('Growth', regime_out['growth_df']),
                                  ('Inflation', regime_out['inflation_df']),
                                  ('Liquidity', regime_out['liquidity_df'])]):
    corr = df.dropna().corr()
    im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr))); ax.set_yticks(range(len(corr)))
    ax.set_xticklabels(corr.columns, rotation=45, ha='right')
    ax.set_yticklabels(corr.columns)
    ax.set_title(f"{name} pillar—input correlations")
    for i in range(len(corr)):
        for j in range(len(corr)):
            ax.text(j, i, f"{corr.values[i,j]:.2f}", ha='center', va='center', fontsize=8)
fig.colorbar(im, ax=axes, fraction=0.02)
fig.savefig(OUT / 'A_pillar_correlations.png', dpi=120, bbox_inches='tight')
plt.show()

## B. Predictive validity — do the scores explain forward asset returns?

OLS at monthly frequency, Newey–West HAC standard errors with `lags = k + 1` to handle overlapping k-month forward returns. The headline is the *sign table* — do the regression coefficients have the signs an economist would expect, and at what significance?

In [ ]:
reg_df = vt.forward_return_regressions(scores, fwd)
reg_df.to_csv(OUT / 'B_ols_full.csv', index=False)

# Headline table: multivariate spec only
mv = reg_df[reg_df['spec'] == 'multivariate:G+I+L'].copy()
mv = mv[['asset','horizon','n','r2','adj_r2',
         'coef_Growth','t_Growth','coef_Inflation','t_Inflation',
         'coef_Liquidity','t_Liquidity']]
print('Multivariate OLS (Growth + Inflation + Liquidity → forward log return)')
print(mv.round(3).to_string(index=False))

In [ ]:
sign_tbl = vt.sign_table(reg_df)
sign_tbl.to_csv(OUT / 'B_sign_table.csv', index=False)

# How often does the actual sign match the prior?
match_rate = sign_tbl['matches_prior'].mean()
sig_match  = sign_tbl[(sign_tbl['matches_prior']) & (sign_tbl['significant_5pct'])]
print(f'Sign matches prior:        {match_rate*100:.1f}% of (asset,horizon,pillar) cells')
print(f'Sign-match AND p<0.05:     {len(sig_match)} of {len(sign_tbl)} cells')
print('\nCells with significant correctly-signed coefficients:')
print(sig_match[['asset','horizon','pillar','coef','t','p','r2','n']].round(3).to_string(index=False))

In [ ]:
# Heatmap of t-stats for the multivariate spec, by asset × pillar × horizon.
pivot = mv.set_index(['asset','horizon'])[['t_Growth','t_Inflation','t_Liquidity']]
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(pivot.values, cmap='RdBu_r', vmin=-3, vmax=3, aspect='auto')
ax.set_yticks(range(len(pivot))); ax.set_yticklabels([f"{a} | {h}M" for a, h in pivot.index])
ax.set_xticks(range(3)); ax.set_xticklabels(['Growth','Inflation','Liquidity'])
ax.set_title('Newey-West t-stats by asset × horizon × pillar  (|t|>2 ≈ 5% significance)')
for i in range(len(pivot)):
    for j in range(3):
        v = pivot.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f'{v:+.2f}', ha='center', va='center', fontsize=9)
fig.colorbar(im, ax=ax)
fig.savefig(OUT / 'B_tstat_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

## C. Recession nowcast — NBER USREC as ground truth

Logit: `P(any USREC=1 in next k months) ~ Growth + Inflation + Liquidity`. Headline metric is AUC; > 0.7 is a reasonable bar for a macro index.

In [ ]:
if usrec is None:
    print('USREC unavailable — skipping recession nowcast')
else:
    nc = vt.recession_nowcast(scores, usrec)
    rows = []
    for k, r in nc.items():
        if 'error' in r:
            print(f'k={k}: {r["error"]}'); continue
        rows.append({'horizon': k, 'n': r['n'], 'base_rate': r['base_rate'],
                     'auc': r['auc'], 'pseudo_r2': r['pseudo_r2'],
                     **{f'coef_{p}': r['coefs'].get(p) for p in ['Growth','Inflation','Liquidity']},
                     **{f'p_{p}':    r['pvalues'].get(p) for p in ['Growth','Inflation','Liquidity']}})
    nowcast_df = pd.DataFrame(rows)
    nowcast_df.to_csv(OUT / 'C_recession_nowcast.csv', index=False)
    print(nowcast_df.round(3).to_string(index=False))

    # Compare against discrete regime-label flag.
    label_signal = vt.regime_label_recession_signal(labels, usrec)
    print('\nDiscrete regime-label flag vs continuous-score logit:')
    for k, r in label_signal.items():
        print(f'  k={k}: AUC(flag)={r.get("auc_label_flag"):.3f}  AUC(logit)={nc[k]["auc"]:.3f}')

In [ ]:
# Plot predicted recession probability (12M) vs actual recessions
if usrec is not None and 'error' not in nc.get(12, {}):
    pred = nc[12]['predicted_prob']
    actual = nc[12]['actual']
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(pred.index, pred.values, color='#d62728', label='Predicted P(recession in next 12M)')
    ax.fill_between(actual.index, 0, actual.values, color='gray', alpha=0.25, step='post', label='Actual NBER recession (next 12M)')
    ax.set_ylim(-0.02, 1.05); ax.set_ylabel('Probability'); ax.set_title('Recession nowcast vs ground truth')
    ax.legend(loc='upper right'); ax.grid(alpha=0.3)
    fig.savefig(OUT / 'C_recession_predicted_vs_actual.png', dpi=120, bbox_inches='tight')
    plt.show()

## D. Regime-conditional return distributions

If the regime label is informative, next-month returns should differ across regimes. Test with one-way ANOVA + Kruskal–Wallis. Use the coarse 6-bucket grouping so per-bucket samples are large enough for the test to be meaningful.

In [ ]:
# Coarse buckets (6) for sample-size reasons; raw 27 in supplemental.
labels_coarse = labels.copy()
labels_coarse['Bucket'] = vt.coarse_regime_groups(labels['Regime'])

rc_coarse = vt.regime_conditional_returns(labels_coarse, prices, group_col='Bucket', min_obs=12)
rc_coarse['summary'].to_csv(OUT / 'D_regime_returns_summary.csv', index=False)
rc_coarse['anova'].to_csv(OUT / 'D_regime_anova.csv')

print('Mean monthly log return by macro bucket (annualized Sharpe in last column):\n')
pivot = rc_coarse['summary'].pivot_table(index='regime', columns='asset',
                                          values='sharpe_annualized')
print(pivot.round(2))
print('\nANOVA & Kruskal-Wallis p-values across buckets:')
print(rc_coarse['anova'].round(4))

In [ ]:
# Bar chart of mean monthly returns by bucket, for SPY
summary = rc_coarse['summary']
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, asset in zip(axes.flat, ['SPY','AGG','GLD','DXY']):
    sub = summary[summary['asset'] == asset].sort_values('mean_log_ret')
    if len(sub) == 0:
        ax.set_visible(False); continue
    yerr = [(sub['mean_log_ret'] - sub['ci_lo']).values,
            (sub['ci_hi'] - sub['mean_log_ret']).values]
    ax.bar(sub['regime'], sub['mean_log_ret']*100, yerr=np.array(yerr)*100, capsize=3,
           color=['#d62728' if v < 0 else '#2ca02c' for v in sub['mean_log_ret']])
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_title(f'{asset} — next-month log return (%) by macro bucket, 95% CI')
    ax.set_xticklabels(sub['regime'], rotation=30, ha='right')
    ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / 'D_regime_returns_bar.png', dpi=120, bbox_inches='tight')
plt.show()

## E. Walk-forward parity check

The 36-month rolling z-score should be causal by construction. Verify by recomputing the regime label as of every 12th month-end using only data available at that date and confirming the result matches the in-sample timeline.

In [ ]:
wf = vt.walk_forward_parity(panel['fred'])
wf.to_csv(OUT / 'E_walk_forward.csv', index=False)
print(f"Walk-forward labels match in-sample at {wf['match'].sum()} / {len(wf)} test dates")
if not wf['match'].all():
    print('\nMismatches:')
    print(wf[~wf['match']].to_string(index=False))
else:
    print('\u2705 Model is point-in-time — no look-ahead.')

## F. Sensitivity analysis

Vary the z-score window for the headline OLS regression and the classification threshold for the discrete-label ANOVA. If the headline conclusions flip across reasonable parameter choices, we have a fragility problem.

In [ ]:
sens_score = vt.sensitivity_grid(panel['fred'], fwd, headline_asset='SPY', headline_horizon=3)
sens_score.to_csv(OUT / 'F_sensitivity_zwindow.csv', index=False)
print('Sensitivity to z-score window (headline: SPY 3M, multivariate G+I+L):')
print(sens_score.round(3).to_string(index=False))

sens_thr = vt.threshold_sensitivity(panel['fred'], prices)
sens_thr.to_csv(OUT / 'F_sensitivity_threshold.csv', index=False)
print('\nSensitivity to classification threshold (ANOVA across coarse buckets):')
print(sens_thr.round(3).to_string(index=False))

## Summary checklist

| Test | Pass criterion | Result |
|------|----------------|--------|
| **A.** Construct validity | PC1 ≥ 40% per pillar, same-signed loadings | see `A_construct_validity.csv` |
| **B.** Predictive validity | ≥ 1 (asset, horizon, pillar) with correct sign and \|t\| > 2 | see `B_sign_table.csv` |
| **C.** Recession nowcast | AUC(12M) > 0.7 | see `C_recession_nowcast.csv` |
| **D.** Regime distinction | ANOVA p < 0.05 on at least one core asset | see `D_regime_anova.csv` |
| **E.** Point-in-time | Walk-forward labels match in-sample | see `E_walk_forward.csv` |
| **F.** Robustness | Headline R² stable to z-window, sign of coefficients preserved | see `F_*.csv` |

**How to read this.** Macro models rarely beat R² of 10–20% on forward returns. The point isn't a high R² — it's that the right pillars predict the right assets in the right direction with statistical confidence. If signs match priors and a non-trivial fraction clear |t| > 2, the framework has substance. If the recession nowcast clears AUC > 0.7, the regime labels meaningfully discriminate the cycle. Together that's the foundation we wanted before building anything on top of the model.